In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameters.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'
PAPER_LOGS_DIR = 'research_notebooks/bowaka_v2_lab/tests/fixtures/paper_logs_minimal'


# 09 — Paper-vs-Backtest Reconciliation

Reconciles paper-trading logs against a **real backtest's** sim artifacts —
candidate, decision, and fill records are matched paper-vs-sim. Point
`PAPER_LOGS_DIR` at real paper logs and `CONFIG_PATH` at a backtest config
covering the same period for a meaningful reconciliation.

In [3]:
import pandas as pd
from bowaka_v2_lab.reconcile import (import_paper_logs, compare_candidates,
  compute_slippage_residuals, render_reconciliation_report)
from bowaka_v2_lab.backtest_runner import run_config_backtest
# 1. paper logs
imp = import_paper_logs(PAPER_LOGS_DIR)
print('paper:', len(imp.candidates), 'candidates,', len(imp.decisions), 'decisions,',
      len(imp.fills), 'fills; drift issues:', len(imp.drift_issues))
# 2. a real backtest provides the sim side
result = run_config_backtest(CONFIG_PATH)
_rd = result.run_dir
sim_candidates = pd.read_parquet(_rd / 'candidate_events.parquet').to_dict('records')
sim_decisions = pd.read_parquet(_rd / 'entry_decisions.parquet').to_dict('records')
sim_fills = pd.read_parquet(_rd / 'fills.parquet').to_dict('records')
print('sim:', len(sim_candidates), 'candidates,', len(sim_decisions), 'decisions,',
      len(sim_fills), 'fills')
# 3. reconcile paper vs sim
cmp_c = compare_candidates(imp.candidates, sim_candidates, window_seconds=120)
cmp_d = compare_candidates(imp.decisions, sim_decisions, window_seconds=120)
residuals = compute_slippage_residuals(imp.fills, sim_fills)
md = render_reconciliation_report(candidate_match=cmp_c, decision_match=cmp_d,
  broker_reject_mismatches=[], slippage_residuals=residuals)
print('candidates:', cmp_c.n_match, 'match /', cmp_c.n_miss, 'miss /', cmp_c.n_extra, 'extra')
print(md[:800])


paper: 4 candidates, 3 decisions, 2 fills; drift issues: 0
sim: 186 candidates, 186 decisions, 6 fills
candidates: 0 match / 4 miss / 186 extra
# Paper-vs-Sim Reconciliation Report

## Candidates
- match: 0
- candidate_miss (paper without sim): 4
- extra (sim without paper): 186

## Entry Decisions
- match: 0
- entry_decision_miss: 3
- extra: 186

## Broker Rejection Mismatches
- broker_rejection_mismatch count: 0

## Slippage
- (no slippage residuals provided)


